# QuestionGen Debug Runner

Batch-first Colab notebook for debugging and testing the current pipeline. Use `runner_ui.ipynb` for the primary staff-facing Gradio launcher.

This notebook keeps direct `run_batch_files(...)`, output preview, and artifact inspection in notebook cells. A Gradio launch hook is included only as an optional debugging add-on at the end.


## 1. Mount Drive And Define Standard Paths


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

# Edit DATA_DIR only if your runtime folder lives somewhere else in Drive.
DATA_DIR = Path("/content/drive/MyDrive/Work/Ebenezer Related/QuestionGenData")
SECRETS_DIR = DATA_DIR / "secrets"
INPUT_DIR = DATA_DIR / "input"
OUTPUT_DIR = DATA_DIR / "output"
LOGS_DIR = DATA_DIR / "logs"

for folder in [SECRETS_DIR, INPUT_DIR, OUTPUT_DIR, LOGS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("DATA_DIR:", DATA_DIR)
print("SECRETS_DIR:", SECRETS_DIR)
print("INPUT_DIR:", INPUT_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)


## 2. Minimal Settings


In [ ]:
API_KEY_FILE = SECRETS_DIR / "api_key.txt"
INPUT_CSV = INPUT_DIR / "questions.csv"
OUTPUT_CSV = OUTPUT_DIR / "generated_questions.csv"
OUTPUT_JSON = OUTPUT_DIR / "generated_questions.json"
OUTPUT_MD = OUTPUT_DIR / "generated_questions.md"
GRADIO_OUTPUT_DIR = OUTPUT_DIR / "gradio"

print("API_KEY_FILE:", API_KEY_FILE)
print("INPUT_CSV:", INPUT_CSV)
print("OUTPUT_CSV:", OUTPUT_CSV)
print("OUTPUT_JSON:", OUTPUT_JSON)
print("OUTPUT_MD:", OUTPUT_MD)
print("Optional Gradio output dir:", GRADIO_OUTPUT_DIR)


## 3. Load Secrets From Drive


In [ ]:
import os
from pathlib import Path

def load_api_keys(filepath: str | Path) -> None:
    filepath = Path(filepath)

    if not filepath.exists():
        raise FileNotFoundError(
            f"API key file not found: {filepath}\n"
            "Create it with lines like OPENAI_API_KEY=sk-..."
        )

    with filepath.open("r", encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()

            if not line or line.startswith("#") or "=" not in line:
                continue

            key, value = line.split("=", 1)
            os.environ[key.strip()] = value.strip()

load_api_keys(API_KEY_FILE)

MODEL_NAME = os.getenv("QUESTIONGEN_MODEL", "gpt-5-mini")
TEMPERATURE = float(os.getenv("QUESTIONGEN_TEMPERATURE", "0"))

print("Loaded env vars:")
print("- OPENAI_API_KEY:", "set" if os.getenv("OPENAI_API_KEY") else "missing")
print("- QUESTIONGEN_MODEL:", MODEL_NAME)
print("- QUESTIONGEN_TEMPERATURE:", TEMPERATURE)


## 4. Advanced Settings

`REPO_BRANCH` must be one of the pushed branches listed in `REPO_BRANCH_OPTIONS`. The active debug launcher currently defaults to `main` and allows only `main` unless you intentionally broaden the allowlist. Colab can pull pushed branches, but it cannot see unpushed local-only edits from this workspace.


In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/AwkAwkAardvark/QuestionGen.git"
REPO_BRANCH_OPTIONS = [
    "main",
]
REPO_BRANCH = REPO_BRANCH_OPTIONS[0]
REPO_DIR = Path("/content/QuestionGen")

if REPO_BRANCH not in REPO_BRANCH_OPTIONS:
    raise ValueError(
        f"REPO_BRANCH must be one of {REPO_BRANCH_OPTIONS}. "
        "Update REPO_BRANCH_OPTIONS only when you intentionally want Colab to target another pushed branch."
    )

print("REPO_URL:", REPO_URL)
print("REPO_BRANCH_OPTIONS:", REPO_BRANCH_OPTIONS)
print("REPO_BRANCH:", REPO_BRANCH)
print("REPO_DIR:", REPO_DIR)


## 5. Clone And Install The Repo


In [ ]:
import importlib
import importlib.util
import shutil
import subprocess
import sys

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run(
    ["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)],
    check=True,
)
get_ipython().run_line_magic("pip", f"install -e '{REPO_DIR}[ui]'")

SRC_DIR = REPO_DIR / "src"
importlib.invalidate_caches()
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

if importlib.util.find_spec("questiongen") is None:
    raise ModuleNotFoundError(
        "questiongen is still not importable after installation. "
        "Check the install cell output before continuing."
    )

print("Cloned and installed branch:", REPO_BRANCH)
print("Repo dir:", REPO_DIR)
print("Source path added:", SRC_DIR)
print("Python executable:", sys.executable)


## 6. Run The Current Batch Pipeline

Broad family selection still comes from `QUESTION_TYPES`, and the batch layer fans each selected family out into one or more subtype rows underneath.


In [ ]:
import json
from collections import Counter

from questiongen.batch import run_batch_files
from questiongen.config import create_structured_llm
from questiongen.graph import compile_question_graph
from questiongen.question_types import QUESTION_TYPES

if not INPUT_CSV.exists():
    raise FileNotFoundError(f"Input CSV not found: {INPUT_CSV}")

question_type_keys = list(QUESTION_TYPES.keys())

runner = compile_question_graph(
    structured_llm_factory=lambda schema: create_structured_llm(
        schema,
        model_name=MODEL_NAME,
        temperature=TEMPERATURE,
    )
)

results = run_batch_files(
    input_csv=str(INPUT_CSV),
    output_csv=str(OUTPUT_CSV),
    question_type_keys=question_type_keys,
    runner=runner,
    output_markdown=str(OUTPUT_MD),
)

OUTPUT_JSON.write_text(
    json.dumps([result.model_dump() for result in results], ensure_ascii=False, indent=2),
    encoding="utf-8",
)

status_counts = Counter(result.status for result in results)
print("Question families:", question_type_keys)
print("Expanded subtype rows:", len(results))
print("Status counts:", dict(status_counts))
print("CSV:", OUTPUT_CSV)
print("JSON:", OUTPUT_JSON)
print("Markdown:", OUTPUT_MD)


## 7. Preview Outputs

The exported CSV and JSON include `QuestionTypeKey`, `QuestionFormatKey`, `QuestionSubtypeKey`, and `QuestionSubtype`, so one broad family may appear multiple times per source row.


In [ ]:
import pandas as pd

df = pd.read_csv(OUTPUT_CSV)
df.head()


In [ ]:
print(OUTPUT_MD.read_text(encoding="utf-8")[:3000])
print(OUTPUT_JSON.read_text(encoding="utf-8")[:3000])


## 8. Optional Gradio Hook

Use this only if you want to inspect the same UI after a batch/debug run. The primary staff path is still `runner_ui.ipynb`, and the upload-vs-Drive-path choice happens inside the Gradio app itself.


In [ ]:
import os

os.environ["QUESTIONGEN_DATA_DIR"] = str(DATA_DIR)
os.environ["QUESTIONGEN_API_KEY_PATH"] = str(API_KEY_FILE)
os.environ["QUESTIONGEN_DRIVE_INPUT_CSV"] = str(INPUT_CSV)
os.environ["QUESTIONGEN_OUTPUT_DIR"] = str(GRADIO_OUTPUT_DIR)

from questiongen.ui.gradio_app import create_app

app = create_app()
app.launch(debug=True)


## 9. Download Outputs


In [ ]:
from google.colab import files

files.download(str(OUTPUT_CSV))
files.download(str(OUTPUT_JSON))
files.download(str(OUTPUT_MD))
